# 03 การทำความสะอาดและเตรียมข้อมูล (Cleaning / Preparation)

**เป้าหมาย:** แปลงข้อมูลดิบเป็นเหตุการณ์ที่มี ACC_CODE ยืนยันได้ โดยทุกแถวที่นำเข้ามีสถานะชัดเจนและไม่มีการทิ้งแบบเงียบ

**อินพุต:** snapshot และ `source_manifest.json` จาก Notebook 01 (ตรวจ SHA256 ก่อนอ่าน)

**การแปลง:**
1. ตรวจ schema ก่อน mapping: ฟิลด์จำเป็น (`required_fields`) ต้องมี alias ตรงอย่างน้อยหนึ่งชื่อ (ยกเว้น ACC_CODE ในปีที่ระบุใน `id_optional_years` คือ 2569) ถ้าไม่ครบจะหยุดพร้อมข้อความบอกวิธีแก้
2. Schema mapping รวม alias ของปี 2569 และปรับช่องว่าง (ตัดหัวท้าย/ยุบช่องว่างซ้ำ) ในทุกฟิลด์ข้อความ รวมถึงคีย์ ID เพื่อไม่ให้ `' 123 '` กับ `'123'` แยกเป็นคนละเหตุการณ์
3. วันที่แบบระมัดระวัง: Excel serial (origin 1899-12-30), ISO `yyyy-mm-dd`, และ `d/m/yyyy` ที่ลองทั้ง D/M และ M/D — ถ้าทั้งสองแบบใช้ได้และต่างกัน = `ambiguous` → กักกัน **ไม่เลือกเอง**; ปีต้องตรงกับปีของไฟล์ (หลักฐานที่มี) มิฉะนั้น `year_mismatch`; บันทึก **วันที่ผู้สมัคร (candidates)** ทุกค่าไว้ใน row audit; เขตเวลา Asia/Bangkok เป็นสมมติฐาน
4. เวลา `HH:MM[:SS]` หรือเศษส่วนวันของ Excel; `nan`/`inf`/`24:00`/ค่าที่ปัดแล้วกลายเป็นเที่ยงคืนวันถัดไป = invalid → กักกัน
5. พิกัดต้องเป็นตัวเลขจำกัดและอยู่ในกรอบกว้างของประเทศไทยเท่านั้น (ไม่ใช่การพิสูจน์ขอบเขตจริง); จังหวัดว่าง = ถูกตัดออก (ไม่ใช่พื้นที่ที่ใช้ได้)
6. กลุ่ม `source_year + ACC_CODE`: รวมเฉพาะแถวที่เหมือนกันทุกฟิลด์หลังปรับมาตรฐาน; ถ้า ID เดียวกันข้อมูลขัดกัน หรือมีสมาชิกที่ไม่ผ่าน → กักกันทั้งกลุ่ม (ไม่เลือกแถวแรกตามอำเภอใจ)
7. แถวไม่มี ACC_CODE → `diagnostic` แยกต่างหาก **คงธง/เหตุผลความผิดปกติไว้** ไม่ใช่เหตุการณ์อิสระที่ยืนยันแล้ว และไม่ใช่ข้อมูลสะอาดทั้งหมด
8. ทุกแถวที่ถูกกักกันหรือเป็น diagnostic จะสร้างธง `possibly_undercounted` ให้ (จังหวัด, วันที่ผู้สมัครทุกค่า) — จังหวัดว่างจะธงวันนั้นทุกจังหวัด

**เอาต์พุต (`data/interim/`):** `cleaned_events.csv`, `events_descriptive_outcomes.csv` (แยกจาก feature), `quarantine.csv`, `diagnostic_rows.csv`, `duplicate_lineage.csv`, `row_audit.csv`, `possible_undercount_flags.csv`, `reconciliation.csv`, `artifact_manifest.json`; และ `reports/data/quality_report.json`

**การตรวจสอบ:** ทุกแถวมีสถานะเดียว (retained/collapsed/quarantined/diagnostic) และรวมแล้วเท่ากับจำนวนแถวดิบต่อปี; ไม่มีวันที่กำกวมใน retained; ไม่มีการเติมค่าที่ไม่ทราบเป็น 0

**ข้อจำกัด:** การกักกันวันที่กำกวมทำให้วันที่ 1–12 ของแต่ละเดือนในไฟล์ slash ถูกนับขาด จึงต้องใช้ธง `possibly_undercounted`; การตัดแถวเวลา/พิกัดผิดออก ทั้งที่อาจทราบวันที่ เป็นการเลือกแบบอนุรักษ์นิยม; จังหวัดที่รายงานไม่ได้ตรวจกับพิกัด; ยังอาจมีปัญหาการรายงานอื่นที่ไม่ทราบ

**สัญญาสำหรับขั้นถัดไป:** Notebook 04 ใช้ `cleaned_events.csv`, `diagnostic_rows.csv`, `possible_undercount_flags.csv` ที่ hash ตรงกับ `data/interim/artifact_manifest.json` เท่านั้น

In [1]:
import os, sys, json, hashlib
from pathlib import Path
import numpy as np, pandas as pd

def find_backend_root():
    env = os.environ.get('ROADAWARE_BACKEND_ROOT')
    if env:
        return Path(env).resolve()
    here = Path.cwd().resolve()
    for c in [here, *here.parents]:
        if (c / 'backend' / 'config' / 'data_pipeline.json').is_file():
            return c / 'backend'
        if c.name == 'backend' and (c / 'config' / 'data_pipeline.json').is_file():
            return c
    raise FileNotFoundError('backend/config/data_pipeline.json not found from ' + str(here) + '; open the repository root or backend/notebooks')

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()

def verify_hashes(base, entries, label):
    bad = []
    for e in entries:
        p = base / e['path']
        if not p.is_file():
            bad.append('missing ' + str(p))
        elif sha256_file(p) != e['sha256']:
            bad.append('sha256 mismatch ' + str(p))
    if bad:
        raise RuntimeError(label + ' failed verification; rerun the producing notebook or use a fresh --output-dir:\n' + '\n'.join(bad))
    return len(entries)

BACKEND = find_backend_root()
REPO = BACKEND.parent
CFG = json.loads((BACKEND / 'config' / 'data_pipeline.json').read_text('utf-8'))
OUT = Path(os.environ.get('ROADAWARE_OUTPUT_DIR') or (BACKEND / CFG['output_root'])).resolve()
if os.environ.get('ROADAWARE_REQUIRE_VENV') == '1':
    assert Path(sys.prefix).resolve() == (REPO / '.venv').resolve(), 'kernel is not the project .venv: ' + sys.executable
RAW, INT, REP = OUT / 'data' / 'raw' / 'accidents', OUT / 'data' / 'interim', OUT / 'reports' / 'data'
for d in (INT, REP):
    d.mkdir(parents=True, exist_ok=True)
man_file = RAW / 'source_manifest.json'
if not man_file.is_file():
    raise FileNotFoundError(str(man_file) + ' missing; run 01_data_collection first')
MAN = json.loads(man_file.read_text('utf-8'))
print('verified raw snapshots:', verify_hashes(OUT, MAN['files'], 'raw snapshot'))

verified raw snapshots: 6


In [2]:
# FUNCTIONS (notebook 03) - self-contained; tests execute this cell directly
import datetime as dt, math, re
import numpy as np, pandas as pd

EXCEL_ORIGIN = dt.date(1899, 12, 30)
EVENT_KEY = ['event_date', 'time_sec', 'lat', 'lon']
DISPOSITIONS = ['retained', 'collapsed', 'quarantined', 'diagnostic']

def norm_value(v):
    return re.sub(r'\s+', ' ', '' if v is None else str(v)).strip()

def norm_text(s):
    return s.fillna('').astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()

def parse_date(raw, expected_year):
    """Return (date or None, rule, status, candidate ISO dates tuple). Never picks between two valid readings."""
    s = norm_value(raw)
    if not s:
        return None, 'none', 'missing', ()
    if re.fullmatch(r'\d+(\.0+)?', s):
        n = int(float(s))
        if not 30000 <= n <= 60000:
            return None, 'excel_serial', 'invalid', ()
        d = EXCEL_ORIGIN + dt.timedelta(days=n)
        return (d if d.year == expected_year else None, 'excel_serial',
                'ok' if d.year == expected_year else 'year_mismatch', (d.isoformat(),))
    m = re.fullmatch(r'(\d{4})-(\d{1,2})-(\d{1,2})', s)
    if m:
        try:
            d = dt.date(*map(int, m.groups()))
        except ValueError:
            return None, 'iso', 'invalid', ()
        return (d if d.year == expected_year else None, 'iso',
                'ok' if d.year == expected_year else 'year_mismatch', (d.isoformat(),))
    m = re.fullmatch(r'(\d{1,2})/(\d{1,2})/(\d{4})', s)
    if m:
        a, b, y = map(int, m.groups())
        if y > 2400:
            y -= 543  # Buddhist-era year
        cands = []
        for dd, mm in ((a, b), (b, a)):
            try:
                d = dt.date(y, mm, dd)
            except ValueError:
                continue
            if d not in cands:
                cands.append(d)
        iso = tuple(sorted(c.isoformat() for c in cands))
        if not cands:
            return None, 'slash', 'invalid', ()
        if len(cands) > 1:
            return None, 'slash_dmy_vs_mdy', 'ambiguous', iso
        d = cands[0]
        rule = 'slash_equal_parts' if a == b else 'slash_single_valid_order'
        return (d if d.year == expected_year else None, rule,
                'ok' if d.year == expected_year else 'year_mismatch', iso)
    return None, 'unrecognized', 'invalid', ()

def parse_time(raw):
    """Return (seconds since midnight or None, status). Never raises on text such as nan/inf."""
    s = norm_value(raw)
    if not s:
        return None, 'missing'
    m = re.fullmatch(r'(\d{1,2}):(\d{2})(?::(\d{2}))?', s)
    if m:
        h, mi, se = int(m.group(1)), int(m.group(2)), int(m.group(3) or 0)
        return (h * 3600 + mi * 60 + se, 'ok') if h < 24 and mi < 60 and se < 60 else (None, 'invalid')
    try:
        x = float(s)
    except ValueError:
        return None, 'invalid'
    if not math.isfinite(x) or not 0 <= x < 1:
        return None, 'invalid'
    sec = int(round(x * 86400))
    if sec >= 86400:
        return None, 'invalid'  # rounds to the next midnight; would silently change the event date
    return sec, 'ok'

def coord_status(lat, lon, bbox):
    la = pd.to_numeric(norm_text(lat), errors='coerce')
    lo = pd.to_numeric(norm_text(lon), errors='coerce')
    finite = pd.Series(np.isfinite(la.to_numpy(dtype=float)) & np.isfinite(lo.to_numpy(dtype=float)), index=lat.index)
    inside = la.between(bbox['lat_min'], bbox['lat_max']) & lo.between(bbox['lon_min'], bbox['lon_max'])
    st = pd.Series('ok', index=lat.index)
    st[~inside] = 'outside_broad_bbox'
    st[~finite] = 'missing_or_nonfinite'
    return la.where(finite), lo.where(finite), st

def validate_schema(columns, aliases, required, id_optional, label):
    """Fail before mapping when an essential raw field is absent or an alias is ambiguous."""
    errors = []
    for canon, names in aliases.items():
        hits = [n for n in names if n in columns]
        if len(hits) > 1:
            errors.append(f'{canon}: several aliases present {hits}')
        if not hits and canon in required and not (canon == 'acc_code' and id_optional):
            errors.append(f'{canon}: none of {names} present')
    if errors:
        raise ValueError(f'{label}: schema check failed before mapping. Add the observed header to schema_aliases in '
                         'backend/config/data_pipeline.json (or fix the snapshot source):\n  ' + '\n  '.join(errors)
                         + f'\n  observed header: {list(columns)}')
    return True

def map_schema(df, aliases):
    out, absent = pd.DataFrame(index=df.index), []
    for canon, names in aliases.items():
        hit = [n for n in names if n in df.columns]
        if hit:
            out[canon] = norm_text(df[hit[0]])
        else:
            out[canon] = ''
            absent.append(canon)
    return out, absent

def row_reasons(rows):
    parts = [np.where(rows[col] == 'ok', '', tag + rows[col].astype(str))
             for col, tag in (('date_status', 'date_'), ('time_status', 'time_'), ('coord_status', 'coord_'))]
    parts.append(np.where(rows['province'] == '', 'province_missing', ''))
    return pd.Series([';'.join(p for p in t if p) for t in zip(*parts)], index=rows.index, dtype=object)

def joined(df, cols):
    s = df[cols[0]].astype(str)
    return s.str.cat([df[c].astype(str) for c in cols[1:]], sep='\x1f') if len(cols) > 1 else s

def assign_dispositions(ev, compare_cols):
    """ev needs source_year, source_row, acc_code, row_reasons + compare_cols.
    Adds disposition, reasons, kept_source_row. Whole ID group is quarantined on any invalid member or conflict."""
    ev = ev.copy()
    ev['acc_code'] = norm_text(ev['acc_code'])
    no_id = ev['acc_code'] == ''
    ev['disposition'] = np.where(no_id, 'diagnostic', '')
    ev['reasons'] = np.where(no_id, np.where(ev['row_reasons'] == '', 'no_acc_code', 'no_acc_code;' + ev['row_reasons']), ev['row_reasons'])
    ev['kept_source_row'] = pd.array([pd.NA] * len(ev), dtype='Int64')
    sub = ev[~no_id]
    if len(sub):
        keys = [sub['source_year'], sub['acc_code']]
        size = sub.groupby(keys)['source_row'].transform('size')
        first = sub.groupby(keys)['source_row'].transform('min')
        invalid = (sub['row_reasons'] != '').groupby(keys).transform('any').astype(bool)
        n_sig = joined(sub, compare_cols).groupby(keys).transform('nunique')
        n_tl = joined(sub, EVENT_KEY).groupby(keys).transform('nunique')
        disp = pd.Series('', index=sub.index, dtype=object)
        reas = sub['row_reasons'].astype(object).copy()
        disp[invalid] = 'quarantined'
        reas[invalid & (sub['row_reasons'] == '')] = 'id_group_member_invalid'
        conflict = ~invalid & (n_sig > 1)
        disp[conflict] = 'quarantined'
        reas[conflict] = np.where(n_tl[conflict] > 1, 'id_conflict_time_location', 'id_conflict_other_fields')
        same = ~invalid & (n_sig == 1)
        disp[same & (sub['source_row'] == first)] = 'retained'
        disp[same & (sub['source_row'] != first)] = 'collapsed'
        ev.loc[sub.index, 'disposition'] = disp
        ev.loc[sub.index, 'reasons'] = reas
        ev.loc[sub.index, 'kept_source_row'] = first.where(same & (size > 1)).astype('Int64')
    return ev

def undercount_flags(rows):
    """(province, candidate date) for every quarantined / diagnostic row; empty province -> '*ALL*'."""
    x = rows[rows['disposition'].isin(['quarantined', 'diagnostic'])]
    recs = [(p if p else '*ALL*', d, r)
            for p, c, r in zip(x['province'], x['date_candidates'], x['reasons'])
            for d in str(c).split(';') if d]
    cols = ['province', 'date', 'n_flagged_records', 'flag_reasons']
    if not recs:
        return pd.DataFrame(columns=cols)
    f = pd.DataFrame(recs, columns=['province', 'date', 'reasons'])
    out = f.groupby(['province', 'date'])['reasons'].agg(
        n_flagged_records='size',
        flag_reasons=lambda r: ';'.join(sorted({t for v in r for t in str(v).split(';') if t}))).reset_index()
    return out[cols]

In [3]:
AL = CFG['schema_aliases']
id_optional = set(CFG['id_optional_years'])
schema_check = []
for f in MAN['files']:
    header = list(pd.read_csv(RAW / f['file'], dtype=str, nrows=0, encoding=f['encoding']).columns)
    validate_schema(header, AL, CFG['required_fields'], str(f['be_year']) in id_optional, f['file'])
    schema_check.append({'be_year': f['be_year'], **{k: next((n for n in v if n in header), '(absent)') for k, v in AL.items()}})
schema_check = pd.DataFrame(schema_check).set_index('be_year').T
schema_check

be_year,2564,2565,2566,2567,2568,2569
year_reported,ปีที่เกิดเหตุ,ปีที่เกิดเหตุ,ปีที่เกิดเหตุ,ปีที่เกิดเหตุ,ปีที่เกิดเหตุ,ปีที่เกิดเหตุ
date_raw,วันที่เกิดเหตุ,วันที่เกิดเหตุ,วันที่เกิดเหตุ,วันที่เกิดเหตุ,วันที่เกิดเหตุ,วันที่เกิดเหตุ
time_raw,เวลา,เวลา,เวลา,เวลา,เวลา,เวลา
report_date_raw,วันที่รายงาน,วันที่รายงาน,วันที่รายงาน,วันที่รายงาน,วันที่รายงาน,วันที่รายงาน
report_time_raw,เวลาที่รายงาน,เวลาที่รายงาน,เวลาที่รายงาน,เวลาที่รายงาน,เวลาที่รายงาน,เวลาที่รายงาน
acc_code,ACC_CODE,ACC_CODE,ACC_CODE,ACC_CODE,ACC_CODE,(absent)
agency,หน่วยงาน,หน่วยงาน,หน่วยงาน,หน่วยงาน,หน่วยงาน,หน่วยงาน
road_agency_type,สายทางหน่วยงาน,สายทางหน่วยงาน,สายทางหน่วยงาน,สายทางหน่วยงาน,สายทางหน่วยงาน,สายทางหน่วยงาน
route_code,รหัสสายทาง,รหัสสายทาง,รหัสสายทาง,รหัสสายทาง,รหัสสายทาง,รหัสสายทาง
route_name,สายทาง,สายทาง,สายทาง,สายทาง,สายทาง,สายทาง


In [4]:
parts, absent_by_file = [], {}
for f in MAN['files']:
    raw = pd.read_csv(RAW / f['file'], dtype=str, keep_default_na=False, encoding=f['encoding'])
    assert len(raw) == f['rows'], f['file']
    c, absent_by_file[f['file']] = map_schema(raw, AL)
    # source_row = 1-based CSV record ordinal with the header as record 1 (first data record = 2);
    # not necessarily the physical line number when a quoted field spans lines.
    c.insert(0, 'source_row', np.arange(2, len(raw) + 2))
    c.insert(0, 'source_year', f['be_year'])
    c.insert(0, 'source_file', f['file'])
    cache = {v: parse_date(v, f['ce_year']) for v in c['date_raw'].unique()}
    c['event_date'] = c['date_raw'].map(lambda v: cache[v][0])
    c['date_rule'] = c['date_raw'].map(lambda v: cache[v][1])
    c['date_status'] = c['date_raw'].map(lambda v: cache[v][2])
    c['date_candidates'] = c['date_raw'].map(lambda v: ';'.join(cache[v][3]))
    tcache = {v: parse_time(v) for v in c['time_raw'].unique()}
    c['time_sec'] = pd.array([tcache[v][0] for v in c['time_raw']], dtype='Int64')
    c['time_status'] = c['time_raw'].map(lambda v: tcache[v][1])
    c['lat'], c['lon'], c['coord_status'] = coord_status(c['latitude'], c['longitude'], CFG['thailand_bbox'])
    c['year_reported_matches_file'] = c['year_reported'] == str(f['ce_year'])
    parts.append(c)
rows = pd.concat(parts, ignore_index=True)
rows['row_reasons'] = row_reasons(rows)
compare = [k for k in AL if k not in ('date_raw', 'time_raw', 'latitude', 'longitude')] + EVENT_KEY
rows = assign_dispositions(rows, compare)
rows['event_uid'] = np.where(rows['disposition'] == 'diagnostic',
                             'diag:' + rows['source_file'] + ':' + rows['source_row'].astype(str),
                             rows['source_year'].astype(str) + ':' + rows['acc_code'])
nonnumeric_counts = {}
for col in CFG['count_fields']:
    num = pd.to_numeric(rows[col].replace('', np.nan), errors='coerce')  # unknown stays NaN, never 0
    nonnumeric_counts[col] = int(((rows[col] != '') & num.isna()).sum())
    rows[col] = num
pd.crosstab(rows['source_year'], rows['disposition'], margins=True)

disposition,diagnostic,quarantined,retained,All
source_year,,,,
2564,0,7463,12599,20062
2565,0,7317,13249,20566
2566,0,10723,13118,23841
2567,0,2164,21788,23952
2568,0,173,23542,23715
2569,15239,0,0,15239
All,15239,27840,84296,127375


In [5]:
lineage = ['event_uid', 'source_file', 'source_year', 'source_row']
core = lineage + ['acc_code', 'event_date', 'time_sec', 'date_rule', 'date_status', 'time_status', 'lat', 'lon', 'coord_status',
                  'province', 'agency', 'road_agency_type', 'route_code', 'route_name', 'km']
ret = rows[rows.disposition == 'retained']
quar = rows[rows.disposition == 'quarantined']
diag = rows[rows.disposition == 'diagnostic'].copy()
diag['row_valid'] = diag['row_reasons'] == ''
flags = undercount_flags(rows)
outputs = {
    'cleaned_events.csv': ret[core],
    'events_descriptive_outcomes.csv': ret[['event_uid'] + CFG['descriptive_outcome_fields']],
    'quarantine.csv': quar[core + ['reasons', 'date_raw', 'date_candidates', 'time_raw', 'latitude', 'longitude']],
    'diagnostic_rows.csv': diag[core + ['row_valid', 'reasons', 'date_candidates']],
    'duplicate_lineage.csv': rows[rows.kept_source_row.notna()][lineage + ['disposition', 'kept_source_row']],
    'row_audit.csv': rows[lineage + ['acc_code', 'disposition', 'reasons', 'date_raw', 'date_rule', 'date_status', 'date_candidates',
                                     'time_raw', 'time_status', 'coord_status', 'province', 'year_reported_matches_file', 'kept_source_row']],
    'possible_undercount_flags.csv': flags,
}
recon = rows.groupby(['source_year', 'disposition']).size().unstack(fill_value=0).reindex(columns=DISPOSITIONS, fill_value=0)
recon['total'] = recon.sum(axis=1)
recon['imported_rows'] = [next(f['rows'] for f in MAN['files'] if f['be_year'] == y) for y in recon.index]
outputs['reconciliation.csv'] = recon.reset_index()
for name, df in outputs.items():
    df.to_csv(INT / name, index=False)

def split_counts(s):
    return pd.Series([t for v in s for t in str(v).split(';') if t], dtype=object).value_counts().to_dict()

unplaceable = rows[rows.disposition.isin(['quarantined', 'diagnostic']) & (rows.date_candidates == '')]
qrep = {
    'reconciliation': recon.reset_index().to_dict('records'),
    'quarantine_reason_counts': split_counts(quar['reasons']),
    'diagnostic_reason_counts': split_counts(diag['reasons']),
    'diagnostic_rows_with_validity_flags': int((~diag['row_valid']).sum()),
    'diagnostic_note': 'no-ID rows are not confirmed independent events; many also carry date/time/coordinate/province flags',
    'date_status_by_year': pd.crosstab(rows.source_year, rows.date_status).reset_index().to_dict('records'),
    'ambiguous_slash_rows_by_year': rows[rows.date_status == 'ambiguous'].groupby('source_year').size().to_dict(),
    'date_rule_counts': rows['date_rule'].value_counts().to_dict(),
    'time_status_counts': rows['time_status'].value_counts().to_dict(),
    'coord_status_counts': rows['coord_status'].value_counts().to_dict(),
    'province_missing_rows_excluded': int((rows['province'] == '').sum()),
    'excluded_rows_without_any_candidate_date_by_year': unplaceable.groupby('source_year').size().to_dict(),
    'flagged_province_days': int(len(flags)), 'flagged_all_province_dates': int((flags['province'] == '*ALL*').sum()),
    'year_reported_mismatch_rows': int((~rows['year_reported_matches_file']).sum()),
    'count_fields_nonnumeric_nonempty': nonnumeric_counts,
    'absent_canonical_fields_by_file': absent_by_file,
    'timezone_assumption': CFG['timezone_assumption'], 'bbox_note': 'broad envelope only, not boundary proof',
    'source_row_definition': MAN.get('source_row_definition'),
    'qualification': 'Excluded/uncertain rows are counted and their candidate province-days flagged; other unknown reporting issues may remain.',
}
(REP / 'quality_report.json').write_text(json.dumps(qrep, ensure_ascii=False, indent=2, default=str), 'utf-8')
art = {'producer': '03_cleaning_preparation', 'input_source_manifest_sha256': sha256_file(man_file),
       'files': [{'path': f'data/interim/{n}', 'sha256': sha256_file(INT / n)} for n in outputs]}
(INT / 'artifact_manifest.json').write_text(json.dumps(art, indent=2), 'utf-8')
recon

disposition,retained,collapsed,quarantined,diagnostic,total,imported_rows
source_year,,,,,,
2564,12599,0,7463,0,20062,20062
2565,13249,0,7317,0,20566,20566
2566,13118,0,10723,0,23841,23841
2567,21788,0,2164,0,23952,23952
2568,23542,0,173,0,23715,23715
2569,0,0,0,15239,15239,15239


In [6]:
assert (recon['total'] == recon['imported_rows']).all(), 'row accounting mismatch'
assert len(rows) == sum(f['rows'] for f in MAN['files'])
assert rows['disposition'].isin(DISPOSITIONS).all(), 'row without disposition'
assert ret['event_uid'].is_unique and (ret['acc_code'] != '').all()
assert (ret['row_reasons'] == '').all() and ret['event_date'].notna().all() and (ret['province'] != '').all()
assert not rows.loc[rows.date_status == 'ambiguous', 'disposition'].isin(['retained', 'collapsed']).any()
assert (rows['acc_code'] == rows['acc_code'].str.strip()).all(), 'unnormalized ID'
col = rows[rows.disposition == 'collapsed']
kept = set(zip(ret['source_year'], ret['acc_code'], ret['source_row']))
assert all((y, a, int(k)) in kept for y, a, k in zip(col['source_year'], col['acc_code'], col['kept_source_row'])), 'collapsed row without retained representative'
mixed = rows[rows.acc_code != ''].groupby(['source_year', 'acc_code'])['disposition'].agg(lambda d: ('quarantined' in set(d)) and bool(set(d) & {'retained', 'collapsed'}))
assert not mixed.any(), 'ID group partly kept and partly quarantined'
flag_keys = set(zip(flags['province'], flags['date']))
need = rows[rows.disposition.isin(['quarantined', 'diagnostic'])]
assert all(((p if p else '*ALL*'), d) in flag_keys for p, c in zip(need['province'], need['date_candidates']) for d in c.split(';') if d)
assert verify_hashes(OUT, art['files'], 'interim artifacts') == len(outputs)
print('Accounting checks passed (these checks do not prove the data are complete or correct).')
print('ambiguous slash rows by BE year:', qrep['ambiguous_slash_rows_by_year'])
print('diagnostic rows:', len(diag), '| of which carry validity flags:', qrep['diagnostic_rows_with_validity_flags'])
print('flagged possibly-undercounted province-days:', qrep['flagged_province_days'])
pd.Series(qrep['quarantine_reason_counts'], name='quarantine reason tokens')

Accounting checks passed (these checks do not prove the data are complete or correct).
ambiguous slash rows by BE year: {2564: 7413, 2565: 7268, 2566: 8295, 2569: 6034}
diagnostic rows: 15239 | of which carry validity flags: 6111
flagged possibly-undercounted province-days: 30379


date_ambiguous                22976
id_conflict_other_fields       2309
province_missing               1952
coord_missing_or_nonfinite      721
coord_outside_broad_bbox          6
Name: quarantine reason tokens, dtype: int64

## ข้อจำกัดและสัญญาสำหรับขั้นถัดไป

- `retained` = เหตุการณ์ที่มี ACC_CODE และผ่านทุกการตรวจ; `collapsed` = แถวซ้ำที่เหมือนกันทุกฟิลด์และชี้ไปยังแถวตัวแทน (`kept_source_row`); `quarantined` = ถูกตัดออกพร้อมเหตุผล; `diagnostic` = ไม่มี ACC_CODE (รวมทั้งไฟล์ 2569) ซึ่ง **ยังคงธงความผิดปกติ** และไม่ใช่ป้ายกำกับ
- แถวที่ถูกตัดออกแต่รู้วันที่ (เช่น เวลา/พิกัดผิด) ถูกตัดแบบอนุรักษ์นิยม และวันนั้นถูกธง `possibly_undercounted`; แถวที่ไม่มีวันที่ผู้สมัครเลยไม่สามารถวางลงวันใดได้ จึงนับแยกไว้ใน `excluded_rows_without_any_candidate_date_by_year`
- ธงเป็นข้อมูลตรวจสอบ (audit) เท่านั้น ห้ามใช้เป็น feature
- การผ่าน assertion หมายถึงบัญชีแถวสอดคล้องกัน ไม่ได้หมายความว่าข้อมูลครบถ้วนหรือถูกต้องทั้งหมด